# 薬学インフォマティクス第5回 <ゲノム配列解析> No. 0

2026.05.07 Kawaguchi RK.

* 内容： pytorchの基本
* 目標： pytorchでのニューラルネットワーク学習の流れを理解する
* リンク： https://github.com/carushi/cb_lab/tree/main/code_collection/2026_pharmacoinformatics/No5

## 参考図書
* Pytorch 自然言語処理プログラミング (新納浩幸)

## Google colab用コード

In [ ]:
!pip install torch pandas seaborn tqdm matplotlib

## Import packages

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

## Preapre the datasets
* iris dataset
* predict iris type (setosa, versicolor, and vernicia)

In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split

iris = datasets.load_iris()
print(iris.data.shape)
print(iris.target)
xtrain, xtest, ytrain, ytest = train_test_split(iris.data, iris.target, test_size=0.5)

(150, 4)
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2]


In [ ]:
xtrain = torch.from_numpy(xtrain).type('torch.FloatTensor')
ytrain = torch.from_numpy(ytrain).type('torch.LongTensor')
xtest = torch.from_numpy(xtest).type('torch.FloatTensor')
ytest = torch.from_numpy(ytest).type('torch.LongTensor')

## Model

In [ ]:
class MyIris(nn.Module):
    def __init__(self):
        super(MyIris, self).__init__()
        self.l1 = nn.Linear(4,6)
        self.l2 = nn.Linear(6,6)
        self.l3 = nn.Linear(6,3)

    def forward(self, x):
        h1 = torch.sigmoid(self.l1(x))
        h2 = self.l2(h1)
        h3 = self.l3(h2)
        return h3


## Training setting

In [ ]:
model = MyIris()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [ ]:
criterion = nn.CrossEntropyLoss()

## Training process

In [ ]:
%%time
model.train()
for i in range(1000):
    output = model(xtrain)
    loss = criterion(output, ytrain)
    if i%50 == 0:
        print('iter:', i, '->', loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

iter: 0 -> 1.1597282886505127
iter: 50 -> 0.9266489744186401
iter: 100 -> 0.5239719152450562
iter: 150 -> 0.3744075298309326
iter: 200 -> 0.24762952327728271
iter: 250 -> 0.16300666332244873
iter: 300 -> 0.11699853092432022
iter: 350 -> 0.09107884764671326
iter: 400 -> 0.07507505267858505
iter: 450 -> 1.1028958559036255
iter: 500 -> 0.06096145510673523
iter: 550 -> 0.053153641521930695
iter: 600 -> 0.047836679965257645
iter: 650 -> 0.04367148131132126
iter: 700 -> 0.04022887349128723
iter: 750 -> 0.03729630634188652
iter: 800 -> 0.034773681312799454
iter: 850 -> 0.09654056280851364
iter: 900 -> 0.03739427030086517
iter: 950 -> 0.032572340220212936
CPU times: user 196 ms, sys: 663 ms, total: 859 ms
Wall time: 293 ms


In [ ]:
torch.save(model.state_dict(), 'myiris.model')
model.load_state_dict(torch.load('myiris.model'))

<All keys matched successfully>

## Output

In [ ]:
model.eval()
# no grad is required to compute the test dataset without optimization
with torch.no_grad():
    output1 = model(xtest)
    ans = torch.argmax(output1, 1) # 0, 1, 2 for each
    print(((ytest == ans).sum().float() / len(ans)).item())

0.9733333587646484


## Batch computation

In [ ]:
%%time
model = MyIris()
n = 75 # data size
bs = 25 # batch size
model.train()
for i in range(1000):
    idx = np.random.permutation(n)
    for j in range(0, n, bs):
        xtm = xtrain[idx[j:min(j+bs, n)]]
        ytm = ytrain[idx[j:min(j+bs, n)]]
        output = model(xtm)
        loss = criterion(output, ytm)
        if i%50 == 0:
            print('iter:', i, 'batch:', j, '->', loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

iter: 0 batch: 0 -> 1.257857322692871
iter: 0 batch: 25 -> 1.1402641534805298
iter: 0 batch: 50 -> 1.0825400352478027
iter: 50 batch: 0 -> 1.1043403148651123
iter: 50 batch: 25 -> 1.12045419216156
iter: 50 batch: 50 -> 1.2558668851852417
iter: 100 batch: 0 -> 1.1521662473678589
iter: 100 batch: 25 -> 1.2086832523345947
iter: 100 batch: 50 -> 1.1198116540908813
iter: 150 batch: 0 -> 1.2753854990005493
iter: 150 batch: 25 -> 1.0861246585845947
iter: 150 batch: 50 -> 1.1191511154174805
iter: 200 batch: 0 -> 1.117617130279541
iter: 200 batch: 25 -> 1.209612488746643
iter: 200 batch: 50 -> 1.1534315347671509
iter: 250 batch: 0 -> 1.1556289196014404
iter: 250 batch: 25 -> 1.1386436223983765
iter: 250 batch: 50 -> 1.1863887310028076
iter: 300 batch: 0 -> 1.0856496095657349
iter: 300 batch: 25 -> 1.1004594564437866
iter: 300 batch: 50 -> 1.2945524454116821
iter: 350 batch: 0 -> 1.1720497608184814
iter: 350 batch: 25 -> 1.2432818412780762
iter: 350 batch: 50 -> 1.0653297901153564
iter: 400 batc

In [ ]:
model.eval()
# no grad is required to compute the test dataset without optimization
with torch.no_grad():
    output1 = model(xtest)
    ans = torch.argmax(output1, 1) # 0, 1, 2 for each
    print(((ytest == ans).sum().float() / len(ans)).item())

0.30666667222976685


## GPU-based computation
* Move tonsor arrays to GPU
* Move the model to GPU

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available()
                      else "cpu")

In [ ]:
torch.backends.mps.is_available() # For mac

True

## Edit the code
* device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
* xtrain = xtrain.to(device)
* ytrain = ytrain.to(device)
* xtest = xtest.to(device)
* ytest = ytest.to(device)
* model = MyIris().to(device)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
xtrain = xtrain.to(device)
ytrain = ytrain.to(device)
xtest = xtest.to(device)
ytest = ytest.to(device)
model = MyIris().to(device)

In [ ]:
%%time
n = 75 # data size
bs = 25 # batch size
model.train()
for i in range(1000):
    idx = np.random.permutation(n)
    for j in range(0, n, bs):
        xtm = xtrain[idx[j:min(j+bs, n)]]
        ytm = ytrain[idx[j:min(j+bs, n)]]
        output = model(xtm)
        loss = criterion(output, ytm)
        if i%50 == 0:
            print('iter:', i, 'batch:', j, '->', loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

iter: 0 batch: 0 -> 1.1462842226028442
iter: 0 batch: 25 -> 1.0734782218933105
iter: 0 batch: 50 -> 1.1622235774993896
iter: 50 batch: 0 -> 1.137373685836792
iter: 50 batch: 25 -> 1.1073763370513916
iter: 50 batch: 50 -> 1.1372359991073608
iter: 100 batch: 0 -> 1.0865535736083984
iter: 100 batch: 25 -> 1.1327039003372192
iter: 100 batch: 50 -> 1.1627287864685059
iter: 150 batch: 0 -> 1.1228030920028687
iter: 150 batch: 25 -> 1.1392793655395508
iter: 150 batch: 50 -> 1.1199036836624146
iter: 200 batch: 0 -> 1.1211143732070923
iter: 200 batch: 25 -> 1.127778172492981
iter: 200 batch: 50 -> 1.1330935955047607
iter: 250 batch: 0 -> 1.1292880773544312
iter: 250 batch: 25 -> 1.1145167350769043
iter: 250 batch: 50 -> 1.1381813287734985
iter: 300 batch: 0 -> 1.165286898612976
iter: 300 batch: 25 -> 1.1353352069854736
iter: 300 batch: 50 -> 1.0813641548156738
iter: 350 batch: 0 -> 1.1427348852157593
iter: 350 batch: 25 -> 1.1225672960281372
iter: 350 batch: 50 -> 1.1166839599609375
iter: 400 ba

In [ ]:
model.eval()
# no grad is required to compute the test dataset without optimization
with torch.no_grad():
    output1 = model(xtest)
    ans = torch.argmax(output1, 1) # 0, 1, 2 for each
    print(((ytest == ans).sum().float() / len(ans)).item())

0.3466666638851166
